# Lakebase 101 — Post-Deploy Permissions

Run this **after** `databricks bundle deploy`.

Grants the app's service principal `CAN_MANAGE_RUN` on synced-table pipelines so the "Sync Now" button works.

In [0]:
CATALOG = "lakebase_101_catalog"
APP_NAME = "lakebase-101-app"

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineAccessControlRequest

w = WorkspaceClient()

# Get the app's service principal name
app = w.apps.get(APP_NAME)
sp_name = app.service_principal_name
print(f"App SP: {sp_name}")

# Find all synced-table pipelines for this catalog and grant permissions
granted = 0
for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"):
    try:
        w.pipelines.set_permissions(
            pipeline_id=p.pipeline_id,
            access_control_list=[
                PipelineAccessControlRequest(
                    service_principal_name=sp_name,
                    permission_level="CAN_MANAGE_RUN"
                )
            ]
        )
        granted += 1
        print(f"  ✅ {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Granted CAN_MANAGE_RUN on {granted} pipeline(s) to {sp_name}")